# Phase 4 — calibration and the one-touch evaluation

Launcher for Phase 4 of Laya-Vision. The protocol in `docs/phase4_protocol.md` was committed before any test run:
temperatures fitted on the 3,000 val images outside the comparison set; the test split read exactly once by
`final_eval.py`; Route A scored on a pre-registered 1,000-image test subsample. Results are reported as they are.
Gate 4: the final results table (test accuracy 0.862, calibrated ECE 0.015).

Needs a GPU (L4). Paths assume the Drive layout in `DECISIONS.md`.

In [ ]:
# Phase 4 — setup: mount Drive, clone the public repo. Needs a GPU (L4 recommended).
import os
os.chdir("/content")

from google.colab import drive
drive.mount("/content/drive")

REPO = "/content/laya-vision"
if not os.path.exists(REPO):
    !git clone -q https://github.com/savita10/laya-vision.git {REPO}
else:
    !git -C {REPO} fetch -q origin && git -C {REPO} reset --hard origin/main
os.chdir(REPO)
!git log --oneline -1

!pip install -q "transformers==4.49.0" "laya==0.3.24" accelerate datasets pillow

D = "/content/drive/MyDrive/laya-vision"
P1, P2, P3, STEP5 = f"{D}/phase1", f"{D}/phase2", f"{D}/phase3", f"{D}/step5"
for p in (P1, P2, P3): os.makedirs(p, exist_ok=True)

import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

## Calibration (val monitor pool, outside the comparison set)
One temperature per view by NLL. Result: combined T 1.108 (ECE 0.025 → 0.022), image T 1.068 (0.023 → 0.011),
text T 1.039 (fitting made it slightly worse → reported at T = 1.0). Both ECE implementations agree.

In [ ]:
!python scripts/calibrate.py --base-ckpt {STEP5}/laya_finetuned --run-dir {P3}/run02_final \
    --shards-root {P2}/shards --records {P1}/splits/val.jsonl --comparison-set /content/laya-vision/splits/val_comparison_2000_seed11.json --out-dir {P3}/calibration

## Val comparison set, three views, calibrated
Like-for-like against Route A on the same 2,000 images: combined 0.847 (ECE 0.019), image-only 0.846, text-only
0.306; Route A 0.501. Also writes per-class accuracy and confusion matrices.

In [ ]:
!python scripts/eval_vision.py --base-ckpt {STEP5}/laya_finetuned --run-dir {P3}/run02_final \
    --shards-root {P2}/shards --records {P1}/splits/val.jsonl --comparison-set /content/laya-vision/splits/val_comparison_2000_seed11.json \
    --temperatures {P3}/calibration/calibration.json --out {P3}/eval_val_final.json
import json
v = json.load(open(f"{P3}/eval_val_final.json"))["views"]["combined"]
for k, a in sorted(v["per_class_accuracy"].items(), key=lambda kv: kv[1]):
    print(f"{k:<26}{a:.3f}")

## Route A on the pre-registered test subsample
`--make-comparison-set` writes `splits/test_comparison_1000_seed17.json` (stratified, seed 17, SHA-256-guarded);
commit it before the scoring run. Route A may read test labels only for this file (recorded exception). ~2 h on an L4.
Result: 0.510 at 7.7 GPU-s/image.

In [ ]:
!python scripts/route_a.py --records {P1}/splits/test.jsonl --comparison-set /content/laya-vision/splits/test_comparison_1000_seed17.json --make-comparison-set
!cp /content/laya-vision/splits/test_comparison_1000_seed17.json {P3}/   # keep a copy on Drive; commit the repo copy before scoring
!python scripts/route_a.py --records {P1}/splits/test.jsonl --comparison-set /content/laya-vision/splits/test_comparison_1000_seed17.json \
    --laya-dir {STEP5}/laya_finetuned --out-dir {P3}/route_a_test

## The one test run
`final_eval.py` is the only script that reads test labels; it refuses to run twice. Result: combined 0.862 (ECE
0.015), image-only 0.858, text-only 0.291; on the same 1,000 ids as Route A, 0.873 vs 0.510.

In [ ]:
!python scripts/final_eval.py --base-ckpt {STEP5}/laya_finetuned --run-dir {P3}/run02_final \
    --shards-root {P2}/shards --records {P1}/splits/test.jsonl \
    --temperatures {P3}/calibration/calibration.json --out-dir {P3}/final

## CPU latency (forward only)
Measures the Laya forward over precomputed embeddings on the Colab CPU (875 ms batch-1 on 12 vCPUs). Note: this
excludes SigLIP; end-to-end CPU latency measured in the local test is ~2.2 s per page (see README).

In [ ]:
!python scripts/eval_vision.py --base-ckpt {STEP5}/laya_finetuned --run-dir {P3}/run02_final \
    --shards-root {P2}/shards --records {P1}/splits/val.jsonl --comparison-set /content/laya-vision/splits/val_comparison_2000_seed11.json \
    --limit 100 --device cpu --views combined --out {P3}/latency_cpu.json

## Stage the release inputs on Drive (for Phase 5)
Collects the Gate 4 evidence (`docs/final/`) and the release inputs (trainable weights, run config, calibration, the
Gate-0 base) into one folder to download to the laptop.

In [ ]:
import shutil, os
S = f"{D}/release_staging"
os.makedirs(f"{S}/docs_final", exist_ok=True); os.makedirs(f"{S}/release_inputs", exist_ok=True)
!cp {P3}/final/final_results.json {P3}/final/*.png {P3}/calibration/calibration.json {P3}/calibration/*.png {P3}/eval_val_final.json {S}/docs_final/
!cp {P3}/run02_final/final/trainable.safetensors {P3}/run02_final/run_config.json {P3}/run02_final/epoch_3/monitor_metrics.json {P3}/calibration/calibration.json {S}/release_inputs/
!cp -r {STEP5}/laya_finetuned {S}/release_inputs/base
!du -sh {S}

Gate 4 recorded in `DECISIONS.md`. Next: Phase 5 (release) runs on the laptop — see `README.md`.